# Phase 2 — Airport Data + Day 5 Transformations
**Date:** 2026-04-14  
**Objective:** Airport Delta Table + Window Functions + Silver Layer Transformations

In [0]:
df = spark.read.option("multiLine", "true").json("s3://airlines-bucket-07860/Airport_data/")
# display(df)

In [0]:
# df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
airlines_schema = StructType([
    StructField("city",      StringType(), True),
    StructField("code",      StringType(), True),
    StructField("country",   StringType(), True),
    StructField("elevation", LongType(),   True),
    StructField("iata",      StringType(), True),
    StructField("icao",      StringType(), True),
    StructField("lat",       DoubleType(), True),
    StructField("lon",       DoubleType(), True),
    StructField("name",      StringType(), True),
    StructField("state",     StringType(), True),
    StructField("tz",        StringType(), True)
])

In [0]:
df = spark.read \
    .option("multiLine", "true") \
    .schema(airlines_schema) \
    .json("s3://airlines-bucket-07860/Airport_data/")
# display(df)

In [0]:
from pyspark.sql.functions import col
df_us = df.filter(col("country") == "US")
# display(df_us)

In [0]:
df_us.count()

13578

In [0]:
df_us.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://airlines-bucket-07860/Airport_data/delta/")

# Day 5 — Silver Layer Transformations

In [0]:
df = spark.read.format("delta").load("s3://airlines-bucket-07860/Bronze_data/flight")
# display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy(
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest"
).orderBy("FlightDate")

df_window = df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

# display(df_window.select(
#     "FlightDate",
#     "Reporting_Airline",
#     "Flight_Number_Reporting_Airline",
#     "Origin",
#     "Dest",
#     "row_num"
# ))

In [0]:
from pyspark.sql.functions import col, lpad, floor, concat, lit

df = df.withColumn(
    "crs_dep_time_std",
    concat(
        lpad(floor(col("CRSDepTime") / 100).cast("int"), 2, '0'),
        lit(":"),
        lpad((col("CRSDepTime") % 100).cast("int"), 2, '0')
    )
).withColumn(
    "crs_arr_time_std",
    concat(
        lpad(floor(col("CRSArrTime") / 100).cast("int"), 2, '0'),
        lit(":"),
        lpad((col("CRSArrTime") % 100).cast("int"), 2, '0')
    )
)

- Hours_to_minutes Function (First Version)
- Function to convert HHMM format to total minutes
- If value is 2400 then return 0

In [0]:

from pyspark.sql.functions import col, when, floor

df = df.withColumn(
    "crs_dep_minutes",
    when(col("CRSDepTime") == 2400, 0)
    .otherwise(
        floor(col("CRSDepTime") / 100) * 60 + (col("CRSDepTime") % 100)
    )
).withColumn(
    "crs_arr_minutes",
    when(col("CRSArrTime") == 2400, 0)
    .otherwise(
        floor(col("CRSArrTime") / 100) * 60 + (col("CRSArrTime") % 100)
    )
)

# display(df.select("CRSDepTime", "crs_dep_minutes", "CRSArrTime", "crs_arr_minutes"))

In [0]:
# hours_to_minutes Function (Second Version — Null Handle)
# Second version - if column value is null then return None
from pyspark.sql.functions import col, when, floor

df = df.withColumn(
    "crs_dep_minutes_safe",
    when(col("CRSDepTime").isNull(), None)   # null handle
    .when(col("CRSDepTime") == 2400, 0)      # special case
    .otherwise(
        floor(col("CRSDepTime") / 100) * 60 + (col("CRSDepTime") % 100)
    )
).withColumn(
    "crs_arr_minutes_safe",
    when(col("CRSArrTime").isNull(), None)
    .when(col("CRSArrTime") == 2400, 0)
    .otherwise(
        floor(col("CRSArrTime") / 100) * 60 + (col("CRSArrTime") % 100)
    )
)

# display(df.select("CRSDepTime", "crs_dep_minutes_safe", "CRSArrTime", "crs_arr_minutes_safe"))

## is_cancelled, is_diverted, is_departure_delay Boolean Columns

In [0]:


from pyspark.sql.functions import col, when

df = df \
    .withColumn(
        "is_cancelled",
        when(col("Cancelled") == 1, True).otherwise(False)) \
    .withColumn(
        "is_diverted",
        when(col("Diverted") == 1, True).otherwise(False)) \
    .withColumn(
        "is_departure_delayed",
        when(col("DepDel15") == 1, True).otherwise(False)) \
    .withColumn(
        "is_arrival_delayed",
        when(col("ArrivalDelayGroups") > 0, True).otherwise(False))

# display(df.select(
#     "Cancelled", "is_cancelled",
#     "Diverted", "is_diverted",
#     "DepDel15", "is_departure_delayed",
#     "ArrivalDelayGroups", "is_arrival_delayed"
# ))

-  Create arrival delay category based on ArrDelayMinutes column
-  if less than 0 = Early
-  0 to 14  = On Time
-  15 to 44 = Minor Delay
-  45 to 120  = Major Delay
-  above 120   = Severe Delay
-  cancelled flight = Cancelled

In [0]:


df = df.withColumn(
    "arrival_delay_category_bucket",
    when(col("Cancelled") == 1, "Cancelled")
    .when(col("ArrDelayMinutes") < 0, "Early")
    .when((col("ArrDelayMinutes") >= 0) & (col("ArrDelayMinutes") <= 14), "On Time")
    .when((col("ArrDelayMinutes") >= 15) & (col("ArrDelayMinutes") <= 44), "Minor Delay")
    .when((col("ArrDelayMinutes") >= 45) & (col("ArrDelayMinutes") <= 120), "Major Delay")
    .when(col("ArrDelayMinutes") > 120, "Severe Delay")
    .otherwise(None)
)

# display(df.select("ArrDelayMinutes", "Cancelled", "arrival_delay_category_bucket"))

-  Date Columns Extract karna
-  Extract date parts from FlightDate column

In [0]:

from pyspark.sql.functions import year, month, quarter, dayofweek, to_date
df = df \
    .withColumn("FlightDate", to_date(col("FlightDate"))) \
    .withColumn("flight_year", year(col("FlightDate"))) \
    .withColumn("flight_month", month(col("FlightDate"))) \
    .withColumn("flight_quarter", quarter(col("FlightDate"))) \
    .withColumn("day_of_week", dayofweek(col("FlightDate")))

# display(df.select("FlightDate", "flight_year", "flight_month", "flight_quarter", "day_of_week"))

In [0]:
# airline_code Column banana
# Create airline_code from IATA_CODE_Reporting_Airline in uppercase
from pyspark.sql.functions import upper

df = df.withColumn(
    "airline_code",
    upper(col("IATA_CODE_Reporting_Airline"))
)

# display(df.select("IATA_CODE_Reporting_Airline", "airline_code"))

In [0]:
# origin_code aur destination_code banana
# Create origin_code and destination_code in uppercase from Origin and Dest
df = df \
    .withColumn("origin_code", upper(col("Origin"))) \
    .withColumn("destination_code", upper(col("Dest")))

# display(df.select("Origin", "origin_code", "Dest", "destination_code"))

In [0]:
# cancellation_reason Column banana
# Create cancellation_reason from CancellationCode
# A = Carrier, B = Weather, C = National Security, D = Security Concern, else = None

df = df.withColumn(
    "cancellation_reason",
    when(col("CancellationCode") == "A", "Carrier")
    .when(col("CancellationCode") == "B", "Weather")
    .when(col("CancellationCode") == "C", "National_Security")
    .when(col("CancellationCode") == "D", "Security_Concern")
    .otherwise(None)
)

# display(df.select("CancellationCode", "cancellation_reason"))

---
# Day 6 — 16 April 2026
**Topics:** Duplicate detection with window functions · load_timestamp · Airport broadcast join

## 6.1 — Duplicate Detection with Window Functions

In [0]:
# Add load_timestamp before dedup so we can order by ingestion time
from pyspark.sql.functions import current_timestamp
df = df.withColumn("load_timestamp", current_timestamp())

In [0]:
# Detect duplicates:
# Partition by the 5 flight-identity keys.
# Order by load_timestamp DESC so row_num=1 = most recently ingested record.
from pyspark.sql.window   import Window
from pyspark.sql.functions import row_number

dedup_window = Window.partitionBy(
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest"
).orderBy(col("load_timestamp").desc())
df = df.withColumn("row_num", row_number().over(dedup_window))
total_before  = df.count()
duplicate_cnt = df.filter(col("row_num") > 1).count()
print(f"Total rows          : {total_before:,}")
print(f"Duplicate rows      : {duplicate_cnt:,}")
print(f"Unique rows to keep : {total_before - duplicate_cnt:,}")

Total rows          : 2,797,629
Duplicate rows      : 0
Unique rows to keep : 2,797,629


In [0]:
# Keep only the first occurrence per flight (row_num == 1)
df = df.filter(col("row_num") == 1).drop("row_num")
print(f"Rows after deduplication: {df.count():,}")

Rows after deduplication: 2,797,629


In [0]:
# display(df)
len(df.columns)

71